# Contexte
Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des
déchets.
Le modèle devra classer chaque image dans l'une des catégories suivantes :

cardboard : cartons ondulés, cartons plats, …

plastic : bouteilles, emballages plastiques...

paper : feuilles, journaux...

glass : bouteilles et objets en verre...

metal : canettes, boîtes métalliques...

trash : emballages bonbons, tasses jetables, ...

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées.

L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning.

In [1]:
# Importation des bibliotheque
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm.auto import tqdm
from PIL import Image
from pathlib import Path


# Partie 1 – Exploration du dataset
Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe,
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa
taille.
NB : prendre en charge aussi les fichiers corrompus

In [2]:
# Liste de tous les chemain de fichiers
RAW_DIR = Path("../data/raw")

fichiers = []
for dossier_classe in sorted(RAW_DIR.iterdir()):
    if dossier_classe.is_dir():
        for fichier in sorted(dossier_classe.iterdir()):
            if fichier.is_file():
                fichiers.append(fichier)

print(len(fichiers), "fichiers trouvés")
print(fichiers[0:2])


1032 fichiers trouvés
[PosixPath('../data/raw/cardboard/cardboard1.jpg'), PosixPath('../data/raw/cardboard/cardboard10.jpg')]


In [10]:
# fonction inspection de l'image
def inspect_image(chemin):
    # 1) La fiche : les infos qu'on peut toujours obtenir, le reste à None
    fiche = {
        "name": chemin.name,
        "class": chemin.parent.name,
        "format": None,
        "mode": None,
        "width": None,
        "height": None,
        "channels": None,
        "pixel_std": None,
        "file_size_kb": round(chemin.stat().st_size / 1024, 2),
        "corrupted": False,
        "error": "",
    }

    # 2) On essaie de lire l'image
    try:
        with Image.open(chemin) as img:
            img.load()                                   # lit tous les pixels
            fiche["format"] = img.format
            fiche["mode"] = img.mode
            fiche["width"] = img.width
            fiche["height"] = img.height
            fiche["channels"] = len(img.getbands())
            gris = np.asarray(img.convert("L"))
            fiche["pixel_std"] = float(gris.std())

    # 3) Si une erreur se produit, l'image est corrompue
    except Exception as e:
        fiche["corrupted"] = True
        fiche["error"] = str(e)[:100]

    return fiche

In [11]:
toutes_les_fiches = []

for f in tqdm (fichiers):
    fiche = inspect_image(f)
    toutes_les_fiches.append(fiche)

print(len(toutes_les_fiches), "fiches creees")

  0%|          | 0/1032 [00:00<?, ?it/s]

1032 fiches creees


In [12]:
df = pd.DataFrame(toutes_les_fiches)
for colonne in ["width", "height","channels"]:
    df[colonne] = df[colonne].astype("Int64")
df.head(636)

,name,class,format,mode,width,height,channels,pixel_std,file_size_kb,corrupted,error
0,cardboard1.jpg,cardboard,JPEG,RGB,512,384,3,31.875892,16.93,False,
1,cardboard10.jpg,cardboard,JPEG,RGB,512,384,3,38.799907,21.17,False,
2,cardboard100.jpg,cardboard,JPEG,RGB,512,384,3,44.498372,14.54,False,
3,cardboard101.jpg,cardboard,JPEG,RGB,512,384,3,68.561937,13.95,False,
4,cardboard102.jpg,cardboard,JPEG,RGB,512,384,3,45.320633,17.59,False,
...,...,...,...,...,...,...,...,...,...,...,...
631,paper211.jpg,paper,JPEG,RGB,512,384,3,29.630460,16.90,False,
632,paper212.jpg,paper,JPEG,RGB,512,384,3,28.926619,11.68,False,
633,paper213.jpg,paper,NaN,NaN,<NA>,<NA>,<NA>,NaN,7.12,True,cannot identify image file '../data/raw/paper/...
634,paper214.jpg,paper,JPEG,RGB,512,384,3,63.676604,47.69,False,
